# Causality Sheet — Cleaning and Final Merge

Cleaning and standardisation of the Causality sheet for the CausalityAI project,
followed by the merge with the case-level, drug-level and reaction-level outputs
prepared elsewhere in the project.

**Contents**

1. Setup and data loading
2. Time-to-onset inspection
3. Dechallenge / Rechallenge splitting
4. Resolving blank and non-standard values
5. Applying the Unknown rule
6. Time-to-onset unit conversion
7. Temporal classification
8. Cross-sheet consistency
9. Composite key verification
10. Deduplication and multi-exposure summary
11. Excluded reports
12. Cleaned Causality output
13. Merge inputs
14. Pre-merge validation
15. The merge
16. Quality check
17. Status

Cleaning is additive throughout: transformations are written to new columns so
that source values stay intact and every derived value remains traceable. Rows
are removed only where a row carries no information not already present in
another, or where a record has been excluded upstream.

## 1. Setup and Data Loading

In [ ]:
import re
import pandas as pd

FILE = "file_new_ids.xlsx"

case_df      = pd.read_excel(FILE, sheet_name="Case")
causality_df = pd.read_excel(FILE, sheet_name="Causality")

# Shared keys used throughout
KEY_COLS = ['report ID', 'WHODrug active ingredient variant', 'MedDRA preferred term']
DRUG_KEY = ['report ID', 'WHODrug active ingredient variant']
PAIR_KEY = ['report ID', 'MedDRA preferred term']

DECH_COL = 'Dechallenge performed? / Reaction resolved/resolving?'
RECH_COL = 'Rechallenge performed? / Reaction recurred?'

print(f"Case sheet:      {case_df.shape[0]:,} rows x {case_df.shape[1]} columns")
print(f"Causality sheet: {causality_df.shape[0]:,} rows x {causality_df.shape[1]} columns")
print(f"Ratio:           {causality_df.shape[0] / case_df.shape[0]:.1f} causality rows per case row")

Each report occupies a single row on the Case sheet but roughly three on the
Causality sheet, since one report may involve several drugs and several
reactions. The Causality sheet therefore sits at the finest level of the three
inputs and serves as the spine of the merge.

## 2. Time-to-onset — Value Inspection

The column is free text combining a number and a unit. The formats actually
present are inspected before any conversion logic is written.

In [ ]:
values = causality_df['Time-to-onset'].unique()
print(f"Distinct formats: {len(values)}")
print(values[:40])

Five units are present — minutes, hours, days, months and years — and a subset of
values carry a negative sign, indicating the reaction predates the recorded
exposure. Both are handled below.

## 3. Dechallenge and Rechallenge — Splitting the Source Columns

Both questions arrive merged into a single cell separated by a slash (for example
`Yes / Yes`). They are separated before any rule is applied to either field.

In [ ]:
causality_df[['Dechallenge_performed', 'Reaction_resolved']] = \
    causality_df[DECH_COL].str.split('/', expand=True)

causality_df[['Rechallenge_performed', 'Reaction_recurred']] = \
    causality_df[RECH_COL].str.split('/', expand=True)

causality_df[['Dechallenge_performed', 'Reaction_resolved',
              'Rechallenge_performed', 'Reaction_recurred']].head()

### Verifying the split

The split assumes exactly one slash per cell. This is confirmed rather than
assumed, since the column heading itself contains a second slash.

In [ ]:
print(causality_df[DECH_COL].astype(str).str.count('/').value_counts())

Every populated cell contains exactly one slash, so the split is sound. Rows not
counted here are entirely empty cells.

### Whitespace

Splitting on the slash leaves a leading space on the second value (`" Yes"`
rather than `"Yes"`). Left uncorrected this causes every string comparison to
fail silently.

In [ ]:
SPLIT_COLS = ['Dechallenge_performed', 'Reaction_resolved',
              'Rechallenge_performed', 'Reaction_recurred']

for col in SPLIT_COLS:
    causality_df[col] = causality_df[col].str.strip()

for col in SPLIT_COLS:
    print(f"--- {col} ---")
    print(causality_df[col].value_counts(dropna=False))
    print()

### Value combinations

Reviewing the combinations of both pairs confirms the value set is consistent and
surfaces any combination needing separate handling.

In [ ]:
print(causality_df.groupby(['Dechallenge_performed', 'Reaction_resolved'],
                           dropna=False).size().sort_values(ascending=False))
print()
print(causality_df.groupby(['Rechallenge_performed', 'Reaction_recurred'],
                           dropna=False).size().sort_values(ascending=False))

## 4. Resolving Blank and Non-standard Values

**Rechallenge blanks.** The field carries no `No` values at all — only `Yes` or
blank — and `Reaction recurred` is blank on exactly the same rows. Blank
therefore records that no rechallenge took place, which is usable information
rather than missing data.

`Reaction recurred` is set to `Not applicable` on those rows rather than `No`,
since asserting the reaction did not recur would imply a re-exposure that never
happened.

In [ ]:
causality_df['Rechallenge_performed'] = causality_df['Rechallenge_performed'].fillna('No')
causality_df['Reaction_recurred']     = causality_df['Reaction_recurred'].fillna('Not applicable')

print(causality_df['Rechallenge_performed'].value_counts(dropna=False))
print()
print(causality_df['Reaction_recurred'].value_counts(dropna=False))

**The dash character.** A dash (U+2012, figure dash) appears in the Dechallenge
fields in place of `Yes` / `No` / `Unknown`. It carries the same meaning as
`Unknown` and is merged into that category.

The replacement matches any dash-like codepoint rather than one exact character,
since the source uses a figure dash rather than a standard hyphen.

In [ ]:
dash_pattern = r'^[\u2010-\u2015\u2212\-]+$'

before = {
    col: causality_df[col].astype(str).str.fullmatch(dash_pattern).sum()
    for col in ['Dechallenge_performed', 'Reaction_resolved']
}

for col in ['Dechallenge_performed', 'Reaction_resolved']:
    causality_df[col] = causality_df[col].replace(dash_pattern, 'Unknown', regex=True)

print(f"Replaced in Dechallenge_performed: {before['Dechallenge_performed']:,}")
print(f"Replaced in Reaction_resolved:     {before['Reaction_resolved']:,}")
print()
print(causality_df['Dechallenge_performed'].value_counts(dropna=False))
print()
print(causality_df['Reaction_resolved'].value_counts(dropna=False))

## 5. Applying the Unknown Rule

Per the agreed cleaning rules, rows where the drug was not withdrawn
(`Dechallenge = No`) while the reaction resolved (`Resolution = Yes`) are
interpreted as `Unknown`.

The rule writes to a derived column `Dechallenge_interpretation`. The source
fields are left unchanged so the raw record stays auditable and the
interpretation can be recomputed if the rule is revised.

In [ ]:
causality_df['Dechallenge_interpretation'] = causality_df['Dechallenge_performed']

rule_mask = (
    (causality_df['Dechallenge_performed'] == 'No') &
    (causality_df['Reaction_resolved']     == 'Yes')
)

print(f"Rows matching the rule: {rule_mask.sum():,}")

causality_df.loc[rule_mask, 'Dechallenge_interpretation'] = 'Unknown'

print()
print(causality_df['Dechallenge_interpretation'].value_counts(dropna=False))

## 6. Time-to-onset — Unit Conversion

All values are converted to days so that intervals become comparable and usable
in analysis.

| Unit | Days |
|---|---|
| 1 minute | 1 / 1440 |
| 1 hour | 1 / 24 |
| 1 day | 1 |
| 1 week | 7 |
| 1 month | 30 |
| 1 year | 365 |

The simplified rule (month = 30, year = 365) is used in preference to the precise
one (30.44 / 365.25); the difference is negligible at this scale and the
simplified form is easier to document and reproduce.

Negative intervals are parsed rather than rejected, so the numeric column covers
every parseable row. They are classified in the next section.

In [ ]:
def time_to_days(value):
    """Convert a Time-to-onset string to days. Accepts negative intervals."""
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    match = re.match(r'^(-?[\d.]+)\s*(minute|hour|day|week|month|year)s?$', value)
    if not match:
        return None

    num  = float(match.group(1))
    unit = match.group(2)

    factors = {'minute': 1/1440, 'hour': 1/24, 'day': 1,
               'week': 7, 'month': 30, 'year': 365}

    return num * factors[unit]


causality_df['Time_to_onset_days'] = pd.to_numeric(
    causality_df['Time-to-onset'].apply(time_to_days), errors='coerce'
)

print(f"Converted rows:    {causality_df['Time_to_onset_days'].notna().sum():,}")
print(f"Of which negative: {(causality_df['Time_to_onset_days'] < 0).sum():,}")

### Verification — unrecognised formats

Any value present in the source column but empty after conversion indicates a
format the function does not handle. Left unchecked this would be silent data
loss.

In [ ]:
unparsed = causality_df[
    causality_df['Time-to-onset'].notna() &
    causality_df['Time_to_onset_days'].isna()
]

print(f"Unparsed rows: {len(unparsed)}")
print(unparsed['Time-to-onset'].unique())

## 7. Time-to-onset — Temporal Classification

Each row is assigned a temporal category following the phenotype-aware framework
set out in the project's systematic review of time-to-onset evidence.

| Category | Range | Handling |
|---|---|---|
| `chronology_impossible` | negative, all negative for the pair | Strong evidence against causality; flag for review |
| `chronology_artefact` | negative alongside non-negative for the pair | Artefact of multi-dose reporting; no evidence against causality |
| `very_early` | < 1 day | Compatible with acute hypersensitivity and infusion reactions |
| `early` | 1–7 days | Compatible with many acute toxicities |
| `intermediate` | 8–56 days | Compatible with delayed hypersensitivity syndromes |
| `delayed` | > 56 days | Compatible with immune-mediated and cumulative toxicities; not auto-rejected |
| `implausibly_long` | > 100 years | Exceeds human lifespan; treated as a data error |
| `missing` | no value | Temporal evidence uninterpretable; lowers model certainty |

The 100-year cut-off is provisional: the review does not specify a numeric
threshold for biological impossibility, so this boundary is open to revision.

In [ ]:
def classify_tto(days):
    if pd.isna(days):
        return 'missing'
    if days < 0:
        return 'chronology_impossible'
    if days > 36500:            # 100 years
        return 'implausibly_long'
    if days < 1:
        return 'very_early'
    if days <= 7:
        return 'early'
    if days <= 56:              # 8 weeks
        return 'intermediate'
    return 'delayed'


causality_df['TTO_category'] = causality_df['Time_to_onset_days'].apply(classify_tto)

ORDER = ['chronology_impossible', 'chronology_artefact', 'very_early', 'early',
         'intermediate', 'delayed', 'implausibly_long', 'missing']

print(causality_df['TTO_category'].value_counts().reindex(ORDER).dropna())

### Distinguishing calculation artefacts from genuine reversals

The review notes that a drug administered on several occasions produces one
interval per exposure, and that first exposure should be distinguished from
re-exposure. The dataset carries no dose-sequence field, so exposure context is
inferred from within-report structure instead.

For each report, drug and reaction group:

- a negative interval alongside one or more non-negative intervals is likely a
  calculation artefact — the reaction began after an early dose and therefore
  precedes later doses of the same drug
- a group whose intervals are all negative indicates the reaction genuinely
  predates the recorded exposure

Only the second case represents an impossible chronology. No rows are removed and
no values are altered.

In [ ]:
group_max        = causality_df.groupby(KEY_COLS)['Time_to_onset_days'].transform('max')
has_non_negative = group_max >= 0
is_neg           = causality_df['Time_to_onset_days'] < 0

causality_df['negative_context'] = pd.NA
causality_df.loc[is_neg &  has_non_negative, 'negative_context'] = 'artefact_multi_dose'
causality_df.loc[is_neg & ~has_non_negative, 'negative_context'] = 'true_reversal'

print(causality_df.loc[is_neg, 'negative_context'].value_counts())
print()

artefact_mask = causality_df['negative_context'] == 'artefact_multi_dose'
causality_df.loc[artefact_mask, 'TTO_category'] = 'chronology_artefact'

counts = causality_df['TTO_category'].value_counts().reindex(ORDER)
print(counts)
print()
print(f"Total: {counts.sum():,}")

### Summary statistics

In [ ]:
converted = causality_df['Time_to_onset_days'].dropna()

print(f"Rows with an interval: {len(converted):,} "
      f"({len(converted) / len(causality_df) * 100:.1f}% of the sheet)")
print(f"Median: {converted.median():.2f} days")
print(f"Mean:   {converted.mean():.2f} days")
print(f"Range:  {converted.min():.2f} to {converted.max():.2f} days")

The median of a few days is consistent with the expectation that most adverse
reactions present shortly after exposure. The gap between median and mean
reflects a small number of long-latency values.

Just under half the sheet carries no interval at all. Under the review's
framework these rows are not discarded; they simply provide no temporal evidence
and should lower model certainty.

### Review file — negative intervals

The negative rows are exported for documentation. They remain in the working
dataset.

In [ ]:
causality_df[is_neg].to_excel("negative_time_to_onset_review.xlsx", index=False)
print(f"Exported {is_neg.sum():,} rows")

## 8. Cross-Sheet Consistency (Case ↔ Causality)

The drug and reaction recorded on the Causality sheet are verified against the
Case sheet for the same report. A mismatch would indicate a linking problem
between the two sheets that must be resolved before the merge.

In [ ]:
case_ids      = set(case_df['report ID'])
causality_ids = set(causality_df['report ID'])

print(f"Unique report IDs in Case:      {len(case_ids):,}")
print(f"Unique report IDs in Causality: {len(causality_ids):,}")
print(f"Present in both:                {len(case_ids & causality_ids):,}")
print(f"Only in Case:                   {len(case_ids - causality_ids):,}")
print(f"Only in Causality:              {len(causality_ids - case_ids):,}")

Drug and reaction values are then compared. The Case sheet stores multiple values
stacked in a single cell, while the Causality sheet holds one value per row, so
the comparison tests whether the Causality value appears within the stacked Case
cell rather than comparing the strings directly.

In [ ]:
def contains_value(stacked, single):
    """Test whether a single value appears within a stacked Case cell."""
    if pd.isna(stacked) or pd.isna(single):
        return None
    parts = str(stacked).replace('_x000D_', '\n').split('\n')
    parts = [p.strip().lower() for p in parts if p.strip()]
    return str(single).strip().lower() in parts


check = case_df[KEY_COLS].merge(
    causality_df[KEY_COLS], on='report ID', suffixes=('_case', '_causality')
)

check['drug_match'] = check.apply(
    lambda r: contains_value(r['WHODrug active ingredient variant_case'],
                             r['WHODrug active ingredient variant_causality']), axis=1)

check['reaction_match'] = check.apply(
    lambda r: contains_value(r['MedDRA preferred term_case'],
                             r['MedDRA preferred term_causality']), axis=1)

print(f"Comparison rows: {len(check):,}")
print()
print(check['drug_match'].value_counts(dropna=False))
print()
print(check['reaction_match'].value_counts(dropna=False))

Drug values match on every row. Reaction values match on all but a small
remainder, and those are not linking errors: the rows are correctly linked by
report, but the Causality sheet holds a non-standard value in the MedDRA field —
either a placeholder where the assessment is not tied to a specific reaction, or
an uncoded free-text term never mapped to a standard MedDRA concept.

Causes observed among the uncoded terms include non-English source text, spelling
errors in the original report, and malformed strings. These represent a
data-quality limitation to record in the final report, not a merge problem.

In [ ]:
unmatched = check[check['reaction_match'] != True]
unmatched.to_excel("meddra_unmatched_review.xlsx", index=False)

print(f"Exported {len(unmatched):,} rows")
print()
print(unmatched['MedDRA preferred term_causality'].value_counts().head(10))

## 9. Composite Key Verification

The cleaning plan proposes merging on report ID plus drug and reaction rather
than report ID alone. That key is tested for uniqueness here, since a non-unique
key inflates row counts during a merge without raising an error.

In [ ]:
key_dup  = causality_df.duplicated(subset=KEY_COLS, keep=False)
full_dup = causality_df.duplicated(keep=False)

print(f"Total rows:                    {len(causality_df):,}")
print(f"Unique key combinations:       {causality_df[KEY_COLS].drop_duplicates().shape[0]:,}")
print(f"Rows sharing a key:            {key_dup.sum():,}")
print(f"  identical on every column:   {full_dup.sum():,}")
print(f"  same key, differing data:    {(key_dup & ~full_dup).sum():,}")
print()

cols_except_time = [c for c in causality_df.columns
                    if c not in ['Time-to-onset', 'Time_to_onset_days',
                                 'TTO_category', 'negative_context']]

dup_without_time = causality_df.duplicated(subset=cols_except_time, keep=False)
conflicting      = key_dup & ~full_dup

print(f"Differing rows:                     {conflicting.sum():,}")
print(f"  identical except Time-to-onset:   {(conflicting & dup_without_time).sum():,}")
print(f"  differing in another field:       {(conflicting & ~dup_without_time).sum():,}")

causality_df[conflicting &  dup_without_time].to_excel("duplicate_keys_time_only.xlsx", index=False)
causality_df[conflicting & ~dup_without_time].to_excel("duplicate_keys_other_diff.xlsx", index=False)

**Finding: the three-column key is not unique.** Rows sharing a key fall into
three groups:

1. **Exact duplicates** across every column — safe to deduplicate without
   information loss.
2. **Differing only in Time-to-onset** — the same report, drug and reaction with
   a different interval, consistent with a drug administered on several
   occasions, each dose producing its own interval against the same reaction.
3. **Differing between `Unknown` and the dash character** — already resolved
   above, since the two carry the same meaning.

The first group is removed in the next section; the second is retained and
summarised.

## 10. Deduplication and Multi-Exposure Summary

### Exact duplicates

Rows identical across every column carry no additional information: the same
report, drug, reaction, interval and assessment recorded more than once.
Retaining them would inflate any downstream count or distribution, and would
weight repeated records more heavily during model training. One copy of each is
kept.

In [ ]:
before    = len(causality_df)
redundant = causality_df.duplicated(keep='first').sum()

causality_df = causality_df.drop_duplicates(keep='first').reset_index(drop=True)

print(f"Rows before:      {before:,}")
print(f"Redundant copies: {redundant:,}")
print(f"Rows after:       {len(causality_df):,}")

### Multi-exposure summary

Rows sharing a key but differing in the recorded interval are retained: each
represents a separate administration and carries genuine information. Rather than
selecting one and discarding the rest, two summary columns are added so that a
one-row-per-key view can be derived later without loss.

| Column | Definition |
|---|---|
| `TTO_first_exposure_days` | the longest non-negative interval in the group |
| `exposure_count` | number of rows recorded for that drug-reaction pair |

The longest non-negative interval corresponds to the earliest administration,
since an earlier dose produces a longer span to the same onset. This matches the
conventional definition of time-to-onset as measured from the start of therapy.

**Assumption:** that rows sharing a key but differing in interval represent
separate administrations. This is inferred from the structure of the data rather
than stated by a dose-sequence field, and is recorded here as an assumption.

In [ ]:
causality_df['_tmp'] = causality_df['Time_to_onset_days'].where(
    causality_df['Time_to_onset_days'] >= 0
)

causality_df['TTO_first_exposure_days'] = (
    causality_df.groupby(KEY_COLS)['_tmp'].transform('max')
)
causality_df['exposure_count'] = (
    causality_df.groupby(KEY_COLS)['report ID'].transform('size')
)
causality_df = causality_df.drop(columns='_tmp')

print("Rows per drug-reaction pair:")
print(causality_df['exposure_count'].value_counts().sort_index().head(10))
print()
multi = (causality_df['exposure_count'] > 1).sum()
print(f"Rows in multi-exposure groups: {multi:,} ({multi / len(causality_df) * 100:.1f}%)")

## 11. Excluded Reports

A number of reports were excluded upstream for an impossible combination of
recorded fields. The same identifiers are removed here so that every part of the
merge covers the same set of reports.

In [ ]:
flags = pd.read_csv("review_flags.csv")

removed_ids = set(
    flags.loc[flags['reason'].str.contains('REMOVED', case=False, na=False), 'report ID']
)
print(f"Report IDs marked REMOVED: {len(removed_ids)}")

before = len(causality_df)
causality_df = causality_df[
    ~causality_df['report ID'].isin(removed_ids)
].reset_index(drop=True)

print(f"Rows before:       {before:,}")
print(f"Rows after:        {len(causality_df):,}")
print(f"Rows removed:      {before - len(causality_df):,}")
print(f"Reports remaining: {causality_df['report ID'].nunique():,}")

## 12. Cleaned Causality Output

Original fields are exported alongside the derived columns so that every
transformation remains traceable to its source value.

`Causality assessment, method, source` is carried through unmodified. It is
cleaned elsewhere in the project and appears here only so each row stays
traceable to its source record.

In [ ]:
OUTPUT_COLS = [
    'report ID',
    'WHODrug active ingredient variant',
    'MedDRA preferred term',
    'Causality assessment, method, source',
    'Time-to-onset',
    'Time_to_onset_days',
    'TTO_category',
    'negative_context',
    'TTO_first_exposure_days',
    'exposure_count',
    'Dechallenge_performed',
    'Reaction_resolved',
    'Dechallenge_interpretation',
    'Rechallenge_performed',
    'Reaction_recurred',
]

output = causality_df[OUTPUT_COLS]
output.to_csv('causality_cleaned.csv', index=False, encoding='utf-8-sig')

print(f"Rows exported:    {len(output):,}")
print(f"Columns exported: {len(output.columns)}")
output.head(3)

## 13. Merge Inputs

The cleaned outputs from the other sections are loaded. Column names and row
counts are confirmed before any join is attempted, since a mismatch in either
produces a silently wrong merge rather than an error.

| Input | Level | Join key |
|---|---|---|
| `case_level_clean.csv` | report | report ID |
| `drug_level_clean.csv` | report + drug | report ID, drug |
| `s2_causality.xlsx` | report + drug + reaction | report ID, drug, reaction |
| `s2_reactions.xlsx` | report + reaction | report ID, reaction |
| `s2_reaction_dates.xlsx` | report + reaction | report ID, reaction |

In [ ]:
case_level   = pd.read_csv("case_level_clean.csv")
drug_level   = pd.read_csv("drug_level_clean.csv")

s2_causality = pd.read_excel("s2_causality.xlsx")
s2_reactions = pd.read_excel("s2_reactions.xlsx")
s2_dates     = pd.read_excel("s2_reaction_dates.xlsx")

inputs = [
    ("case_level",   case_level,   ['report ID']),
    ("drug_level",   drug_level,   DRUG_KEY),
    ("s2_causality", s2_causality, KEY_COLS),
    ("s2_reactions", s2_reactions, PAIR_KEY),
    ("s2_dates",     s2_dates,     PAIR_KEY),
]

for name, df, key in inputs:
    missing = [c for c in key if c not in df.columns]
    print(f"--- {name}: {len(df):,} rows ---")
    if missing:
        print(f"  MISSING KEY COLUMNS: {missing}")
    else:
        print(f"  duplicate keys: {df.duplicated(subset=key, keep=False).sum():,}")
    print(f"  columns: {df.columns.tolist()}")
    print()

## 14. Pre-merge Validation

Two conditions are checked before joining.

**Key uniqueness.** A non-unique key on the right-hand side of a join multiplies
rows silently rather than raising an error.

**Identifier type.** A key read as text in one input and as a number in another
fails to match even where the values are identical, and the join reports no
error when it does.

In [ ]:
print("report ID dtypes")
for name, df, _ in inputs:
    print(f"  {name:<14} {df['report ID'].dtype}")
print(f"  {'causality':<14} {causality_df['report ID'].dtype}")

### Collapsing the drug-level input

The drug-level input holds several records for the same report and drug where a
drug was administered more than once. Joining it directly would multiply rows
without adding information, since no shared field links a given interval on the
Causality side to a specific administration record: every combination would be
produced, most of them spurious.

One record is therefore selected per report and drug — the earliest by recorded
start date, falling back to the lowest recorded index where no start date is
available. This matches the first-exposure basis already used for the interval
columns, so both sides describe the same administration.

Ordering by index alone was tested and found unreliable: of the consecutive pairs
where both start dates are known, a material share run backwards, and a large
share of start dates are not recorded at all.

`drug_record_count` preserves how many records existed for each pair.

In [ ]:
d = drug_level.copy()
d['_start'] = pd.to_datetime(d['Drug start date'], errors='coerce')
d = d.sort_values(DRUG_KEY + ['_start', 'drug_index'], na_position='last')

record_counts = d.groupby(DRUG_KEY).size().rename('drug_record_count')

drug_first = (
    d.drop_duplicates(subset=DRUG_KEY, keep='first')
     .drop(columns='_start')
     .merge(record_counts, on=DRUG_KEY)
)

print(f"drug_level rows before: {len(drug_level):,}")
print(f"After collapsing:       {len(drug_first):,}")
print(f"Unique on key:          {not drug_first.duplicated(subset=DRUG_KEY).any()}")

## 15. The Merge

The Causality sheet is the spine. The other inputs are joined onto it from the
left, so the row count is governed by the spine and cannot grow.

Suffixes keep the spine's own columns unrenamed where a name appears on both
sides, so existing references remain valid.

The assessment input carries the WHO-UMC subset only and therefore covers a
fraction of the spine. The remaining rows hold no assessment; that is expected
and is not a merge failure.

In [ ]:
final = causality_df.copy()

joins = [
    ("drug_level",   drug_first,   DRUG_KEY,     '_drug'),
    ("case_level",   case_level,   ['report ID'], '_case'),
    ("s2_causality", s2_causality, KEY_COLS,     '_s2'),
    ("s2_reactions", s2_reactions, PAIR_KEY,     '_rx'),
    ("s2_dates",     s2_dates,     PAIR_KEY,     '_dt'),
]

for name, right, key, suffix in joins:
    final = final.merge(right, on=key, how='left',
                        suffixes=('', suffix), indicator='_m')
    matched = (final['_m'] == 'both').sum()
    print(f"{name:<14} rows: {len(final):>9,}  matched: {matched:>9,} "
          f"({matched / len(final) * 100:5.1f}%)")
    final = final.drop(columns='_m')

print()
print(f"Spine rows:    {len(causality_df):,}")
print(f"Final rows:    {len(final):,}")
print(f"Final columns: {len(final.columns)}")

## 16. Quality Check

Three controls are applied to the merged table: row-count reconciliation, key
integrity, and completeness by column.

Row-count reconciliation is the primary control. A left join onto a spine cannot
lose rows, and cannot add them when the right-hand input is unique on its key, so
any deviation from the spine count indicates a fault in the merge rather than in
the data.

In [ ]:
print("Row count reconciliation")
print(f"  Spine:      {len(causality_df):,}")
print(f"  Merged:     {len(final):,}")
print(f"  Difference: {len(final) - len(causality_df):+,}")
print()

print("Coverage")
print(f"  Distinct reports:   {final['report ID'].nunique():,}")
print(f"  Distinct drugs:     {final['WHODrug active ingredient variant'].nunique():,}")
print(f"  Distinct reactions: {final['MedDRA preferred term'].nunique():,}")
print()

print("Key integrity — nulls must be zero")
for c in KEY_COLS:
    print(f"  {c}: {final[c].isna().sum():,}")
print()

missing = (final.isna().mean() * 100).sort_values(ascending=False)
print("Completeness — most incomplete columns (% missing)")
print(missing.head(15).round(1).to_string())

Several columns are highly incomplete by design rather than by defect: flag
fields populated only when the condition applies, and derived columns populated
only for the subset they describe. These should not be read as missing data in
the quality report.

The genuinely incomplete fields are the temporal ones — recorded intervals and
drug start dates — which the systematic review identifies as a recognised
limitation of spontaneous reporting data and a reason to lower model certainty
rather than to discard rows.

### Value Distribution by Column

The distribution of each column is reviewed alongside the completeness figures
above. Categorical columns are summarised by their value counts; numeric columns
by their range and central tendency. High-cardinality text columns are reported
by their number of distinct values only, since listing them adds no information.

The purpose is to surface values that are individually valid but collectively
implausible — a category that dominates unexpectedly, or a range that extends
beyond what the field can hold.

In [ ]:
import numpy as np

for col in final.columns:
    n_unique = final[col].nunique(dropna=True)

    if pd.api.types.is_numeric_dtype(final[col]) and n_unique > 20:
        s = final[col].dropna()
        if len(s):
            print(f"--- {col}  (numeric, {n_unique:,} distinct) ---")
            print(f"    min {s.min():,.2f}  |  median {s.median():,.2f}  "
                  f"|  mean {s.mean():,.2f}  |  max {s.max():,.2f}")
            print()

    elif n_unique <= 20:
        print(f"--- {col}  ({n_unique} distinct) ---")
        print(final[col].value_counts(dropna=False).to_string())
        print()

    else:
        top = final[col].value_counts().head(3)
        print(f"--- {col}  ({n_unique:,} distinct — top 3 shown) ---")
        print(top.to_string())
        print()

In [ ]:
final.to_csv('merged_final.csv', index=False, encoding='utf-8-sig')

print(f"Rows:    {len(final):,}")
print(f"Columns: {len(final.columns)}")
print(f"Reports: {final['report ID'].nunique():,}")

## 17. Status

**Complete**

- Time-to-onset standardised, classified, and negative intervals characterised
- Dechallenge / Rechallenge fields split, blanks and non-standard values resolved
- Agreed Unknown rule applied to a derived column
- Cross-sheet consistency verified against the Case sheet
- Composite key tested; exact duplicates removed and multi-exposure groups summarised
- Excluded reports removed in line with the other sections
- All five inputs merged onto the Causality spine with the row count preserved
- Quality checks applied to the merged table

**Open points carried forward**

- The threshold separating a delayed onset from a biologically impossible one is
  provisional and should be confirmed.
- The multi-exposure and multi-administration columns are retained rather than
  collapsed, so the merged table remains at one row per recorded interval. A
  one-row-per-pair view can be derived from the summary columns if required.
- Assessment coverage is limited to the WHO-UMC subset, so the analysable portion
  of the merged table is substantially smaller than its row count.

**Files produced**

| File | Contents |
|---|---|
| `causality_cleaned.csv` | the cleaned Causality sheet |
| `merged_final.csv` | the merged table |
| `negative_time_to_onset_review.xlsx` | negative intervals |
| `meddra_unmatched_review.xlsx` | rows with non-standard MedDRA values |
| `duplicate_keys_time_only.xlsx` | shared key, differing interval |
| `duplicate_keys_other_diff.xlsx` | shared key, differing elsewhere |